# Prepare Granite for GPU Serving with LLM Compressor

Your team plans to serve the IBM Granite 4.0 350M model on a Red Hat OpenShift AI cluster with GPUs.
Before you request GPU resources, prepare two quantized variants of the model, confirm that both variants still produce useful answers, and estimate how much GPU memory (VRAM) each variant needs.

You create the following variants:

* **W8A16**: 8-bit integer weights, 16-bit activations.
* **W4A16**: 4-bit integer weights, 16-bit activations.

Both schemes are *data-free*: LLM Compressor quantizes the weights directly, without a calibration dataset.
This workbench has no GPU, so you run the quantization and the quality checks on CPU.
The quantized models are ready to be deployed to GPU-backed serving runtimes later.

### 1. Import libraries

In [ ]:
import json
import subprocess
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

print("✓ Libraries imported")

### 2. Configure paths and test prompts

The original model is mounted read-only from shared storage.
The quantized models are saved in the `quantized-models` directory, next to this notebook.
The test prompts check general knowledge, reasoning, and code generation.

In [ ]:
MODEL_PATH = Path("/shared-models/granite-4.0-350m")
OUTPUT_DIR = Path("quantized-models")

PROMPTS = [
    "The capital of France is",
    "Explain in one sentence why the sky is blue.",
    "Write a Python function that returns the square of a number.",
]

results = {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("✓ Configuration ready")

### 3. Define helper functions

* `load_model()` loads a model from disk.
* `generate()` sends each test prompt to a model and returns the answers.
* `weights_size_mb()` returns the size of the model weights files (`*.safetensors`).
* `quantize_and_save()` applies a recipe to a fresh copy of the original model, saves the compressed model, and then reloads the saved model from disk to verify that it works.

In [ ]:
def load_model(path):
    return AutoModelForCausalLM.from_pretrained(path, dtype="auto", low_cpu_mem_usage=True)


def generate(model, max_new_tokens=40):
    answers = []
    for prompt in PROMPTS:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return answers


def weights_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).glob("*.safetensors")) / 1024**2


def print_answers(answers):
    for prompt, answer in zip(PROMPTS, answers):
        print(f"Q: {prompt}\nA: {answer}\n")


def quantize_and_save(name, recipe):
    if recipe is None:
        raise ValueError(f"Complete the TODO in the {name} recipe cell, run it, and then run this cell again.")
    save_dir = OUTPUT_DIR / f"granite-4.0-350m-{name}"

    model = load_model(MODEL_PATH)
    oneshot(model=model, recipe=recipe)
    model.save_pretrained(save_dir, save_compressed=True)
    tokenizer.save_pretrained(save_dir)
    del model

    # Reload the saved model to verify that the compressed files work
    reloaded = load_model(save_dir)
    answers = generate(reloaded)
    del reloaded

    results[name] = {"path": save_dir, "size_mb": weights_size_mb(save_dir), "answers": answers}
    print(f"✓ {name} model saved to {save_dir}/ ({results[name]['size_mb']:.0f} MB)\n")
    print_answers(answers)


print("✓ Helper functions defined")

### 4. Test the original model

Record the size and the answers of the original BF16 model as the baseline for the comparison.

In [ ]:
original = load_model(MODEL_PATH)
results["BF16"] = {
    "path": MODEL_PATH,
    "size_mb": weights_size_mb(MODEL_PATH),
    "answers": generate(original),
}
del original

print(f"✓ Original model: {results['BF16']['size_mb']:.0f} MB\n")
print_answers(results["BF16"]["answers"])

### 5. Recipe 1: W8A16

A recipe tells LLM Compressor which layers to quantize and how.
The `QuantizationModifier` class takes the following parameters:

* `targets`: the layer types to quantize. `"Linear"` selects all linear layers, which hold most of the model weights.
* `scheme`: the quantization scheme, such as `"W8A16"` or `"W4A16"`.
* `ignore`: layers to keep at the original precision.
This model shares the weights of its output layer, `lm_head`, with the input embeddings, so keep `lm_head` at full precision.

Complete the TODO to set the scheme for 8-bit integer weights and 16-bit activations.

In [ ]:
recipe_w8a16 = None

# TODO: replace the previous line with a recipe that quantizes all Linear layers
# with the W8A16 scheme and ignores the lm_head layer:
#
# recipe_w8a16 = QuantizationModifier(
#     targets="Linear",
#     scheme="...",
#     ignore=["lm_head"],
# )

In [ ]:
quantize_and_save("W8A16", recipe_w8a16)

### 6. Recipe 2: W4A16

Write the second recipe yourself.
Use the same `targets` and `ignore` values as the W8A16 recipe, with the scheme for 4-bit integer weights and 16-bit activations.

In [ ]:
recipe_w4a16 = None

# TODO: replace the previous line with a QuantizationModifier recipe that quantizes
# all Linear layers with the W4A16 scheme and ignores the lm_head layer.

In [ ]:
quantize_and_save("W4A16", recipe_w4a16)

### 7. Estimate GPU memory

In a serving engine (for example vLLM on GPU), the usual production case: the weights stay packed in GPU memory, and special code works on them directly.
W8A16 roughly halves the memory the weights take, and W4A16 roughly quarters it.
That's the main reason people quantize: a model that didn't fit on a GPU now fits, or leaves room for more simultaneous requests.
It's also faster, because generating text is limited mostly by how quickly the weights can be read from memory, and smaller weights are read faster.

Estimate the VRAM that each model needs as the sum of two parts:

* **Weights**: the packed weights occupy about the same space in GPU memory as on disk.
* **KV cache**: the attention keys and values of every token that the model is processing.
The KV cache stays at 16-bit precision for all three models, because W8A16 and W4A16 quantize only the weights.

KV cache bytes = 2 (keys and values) × attention layers × KV heads × head dimension × 2 bytes × tokens

The estimate assumes the workload in the following cell.
Real deployments also need memory for activations and for the serving runtime itself.

In [ ]:
CONCURRENT_REQUESTS = 8
TOKENS_PER_REQUEST = 4096

config = json.loads((MODEL_PATH / "config.json").read_text())
attention_layers = config.get("layer_types", ["attention"] * config["num_hidden_layers"]).count("attention")
head_dim = config["hidden_size"] // config["num_attention_heads"]
kv_bytes_per_token = 2 * attention_layers * config["num_key_value_heads"] * head_dim * 2
kv_cache_gb = kv_bytes_per_token * CONCURRENT_REQUESTS * TOKENS_PER_REQUEST / 1024**3

print(f"KV cache per token: {kv_bytes_per_token / 1024:.0f} KiB")
print(f"KV cache for {CONCURRENT_REQUESTS} requests x {TOKENS_PER_REQUEST} tokens: {kv_cache_gb:.2f} GB")

### 8. Compare the models

Compare the size, the estimated VRAM, and the answers of the three models.

In [ ]:
baseline_mb = results["BF16"]["size_mb"]

print(f"{'Model':<8}{'Weights (MB)':>14}{'Reduction':>11}{'Weights VRAM (GB)':>19}{'Est. VRAM (GB)':>16}")
print("-" * 68)
for name, r in results.items():
    weights_gb = r["size_mb"] / 1024
    reduction = (1 - r["size_mb"] / baseline_mb) * 100
    print(f"{name:<8}{r['size_mb']:>14.0f}{reduction:>10.1f}%{weights_gb:>19.2f}{weights_gb + kv_cache_gb:>16.2f}")

for i, prompt in enumerate(PROMPTS):
    print(f"\nQ: {prompt}")
    for name, r in results.items():
        answer = " ".join(r["answers"][i].split())
        print(f"  {name:<6} {answer[:100]}")

### 9. Scale the estimate to a larger model

The 350M model is small, so its KV cache is larger than its weights.
For large models, the weights dominate, and quantization decides how many GPUs you need.
Change `PARAMETERS` to estimate the weights VRAM of other model sizes.

In [ ]:
PARAMETERS = 70e9  # 70B parameters

for name, bits in [("BF16", 16), ("W8A16", 8), ("W4A16", 4)]:
    print(f"{name:<6} ~{PARAMETERS * bits / 8 / 1024**3:>6.0f} GB of weights")